# Step 1: Hyperparameter Search

This notebook performs a **grid search** over neural network hyperparameters to find
the best configuration for detecting nonlinear Granger causality.

**Pipeline**: `01_hyperparameter_search` → 02_run_experiment → 03_results_analysis

---

In [ ]:
# Add project root to path
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import src  # Sets TF_USE_LEGACY_KERAS

import numpy as np
import pickle
import time
import gc
from datetime import datetime
from itertools import product

from src.config import NN_CONFIG_MAP, get_direction_labels, validate_map, validate_direction, validate_architecture
from src.data import generate_data, normalize_data, split_data
from src.causality import run_causality_test

print("Imports successful.")

## Configuration

Modify the parameters below to set up the grid search.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Chaotic map: "henon", "ikeda", "tinkerbell", "rulkov"
CHAOTIC_MAP = "henon"

# Causality direction:
#   "Y_to_X" — tests whether Y Granger-causes X
#   "X_to_Y" — tests whether X Granger-causes Y
CAUSALITY_DIRECTION = "Y_to_X"

# NN architectures to evaluate: "MLP", "LSTM", "GRU"
NN_ARCHITECTURES = ["GRU"]

# Hyperparameter search space
NEURONS_OPTIONS = [10, 50, 100]
LAG_OPTIONS = [5, 10, 20]
BATCH_SIZE_OPTIONS = [16, 32]
EPOCHS = 50
LEARNING_RATE = 0.0001

# Time series length
N_ITERATIONS = 500

# Data splits
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2

# Output directory
OUTPUT_DIR = "../results/hyperparameters/"

# =============================================================================
# Validation
validate_map(CHAOTIC_MAP)
validate_direction(CAUSALITY_DIRECTION)
for arch in NN_ARCHITECTURES:
    validate_architecture(arch)

labels = get_direction_labels(CAUSALITY_DIRECTION)

combos = list(product(NN_ARCHITECTURES, NEURONS_OPTIONS, LAG_OPTIONS, BATCH_SIZE_OPTIONS))
print(f"Map: {CHAOTIC_MAP} | Direction: {labels['direction_str']}")
print(f"Total combinations: {len(combos)}")

## Data Generation

In [ ]:
# Generate, normalize, and split data
data = generate_data(CHAOTIC_MAP, CAUSALITY_DIRECTION, N_ITERATIONS)
data = normalize_data(data)
data_train, data_val, data_test = split_data(data, TRAIN_RATIO, VAL_RATIO)

print(f"Data shape: {data.shape}")
print(f"Train: {data_train.shape} | Val: {data_val.shape} | Test: {data_test.shape}")
print(f"Normalized range: [{data.min():.4f}, {data.max():.4f}]")

## Grid Search

In [ ]:
best_error = float("inf")
best_config = None
all_results = {}

start_time = time.time()

for idx, (arch, neurons, lag, bs) in enumerate(combos, 1):
    print(f"\nConfig {idx}/{len(combos)}: Arch={arch}, Neurons={neurons}, Lag={lag}, BS={bs}")

    metrics, error_msg = run_causality_test(
        data_train, data_val, data_test,
        lag=lag, architecture=arch, neurons=neurons,
        epochs=EPOCHS, learning_rate=LEARNING_RATE, batch_size=bs,
    )

    entry = {
        "nn_architecture": arch, "neurons": neurons,
        "lag": lag, "batch_size": bs,
    }

    if metrics is not None:
        entry.update({
            "p_value": metrics["p_value"],
            "total_error": metrics["total_error"],
            "error": "none",
        })
        if metrics["p_value"] < 0.05 and metrics["total_error"] < best_error:
            best_error = metrics["total_error"]
            best_config = entry.copy()
        print(f"  p={metrics['p_value']:.4f}, error={metrics['total_error']:.6f}")
    else:
        entry["error"] = error_msg
        print(f"  ERROR: {error_msg}")

    all_results[idx] = entry
    gc.collect()

elapsed = time.time() - start_time
print(f"\nGrid search complete in {elapsed:.1f}s")

## Save Results

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

final = {
    "config": {
        "chaotic_map": CHAOTIC_MAP,
        "causality_direction": CAUSALITY_DIRECTION,
        "direction_str": labels["direction_str"],
        "target_variable": labels["target_label"],
        "cause_variable": labels["cause_label"],
        "nn_architectures": NN_ARCHITECTURES,
        "neurons_options": NEURONS_OPTIONS,
        "lag_options": LAG_OPTIONS,
        "batch_size_options": BATCH_SIZE_OPTIONS,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "n_iterations": N_ITERATIONS,
        "total_combinations": len(combos),
        "timestamp": datetime.now().isoformat(),
        "execution_time_seconds": elapsed,
    },
    "all_results": all_results,
    "best_config": best_config,
}

filename = f"hyperparameters_{CHAOTIC_MAP}_{CAUSALITY_DIRECTION}.pkl"
filepath = os.path.join(OUTPUT_DIR, filename)

with open(filepath, "wb") as f:
    pickle.dump(final, f)

print(f"Results saved: {filepath}")

if best_config:
    print(f"\nBest configuration (p < 0.05):")
    for k, v in best_config.items():
        print(f"  {k}: {v}")
else:
    print("\nNo configuration achieved p < 0.05.")